# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-enginee

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
Do not include social media links, such as LinkedIn, Twitter/X, Facebook, Instagram or YouTube.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.
    
    Links (some might be relative URLs):
    
    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


    Here is the list of links on the website https://edwarddonner.com -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative URLs):

    #wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-archi

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content":
            get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for URL: {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model = MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for URL: https://edwarddonner.com by calling gpt-5-nano
Found 3 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    page_content = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{page_content}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link['url'])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for URL: https://huggingface.co by calling gpt-5-nano
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4.1-Flash-1017
Updated
5 days ago
•
51.9k
•
2.32k
black-forest-labs/flux-3-image-klein-comfy
Updated
5 days ago
•
513k
•
1.27k
meta-models/Muse-Spark-Open-816B-A42B
Updated
7 days ago
•
3.16k
•
653
OpenMOSS-Team/MOSS-Image-SFT
Updated
1 day ago
•
6.89k
•
559

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
    You are looking at a company called: {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for URL: https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


'\n    You are looking at a company called: HuggingFace\n    Here are the contents of its landing page and other relevant pages;\n    use this information to build a short brochure of the company in markdown without code blocks.\n\n\n    ## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4.1-Flash-1017\nUpdated\n5 days ago\n•\n51.9k\n•\n2.32k\nblack-forest-labs/flu

In [16]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ]
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for URL: https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future. Serving as a global collaboration platform for the machine learning community, Hugging Face empowers engineers, scientists, and AI enthusiasts to create, discover, and share machine learning models, datasets, and applications. The platform hosts over **2 million models**, **500k+ datasets**, and **1 million+ applications**, making it the hub of open and ethical AI innovation.

At the heart of the AI revolution, Hugging Face provides tools and libraries that accelerate AI development and foster collaboration to build the next generation of machine learning solutions.

---

## What We Offer

- **Models Hub:** Access and contribute to millions of pre-trained ML models in various domains such as natural language processing, computer vision, and more.
- **Datasets:** A vast repository of curated public datasets facilitating efficient training and evaluation.
- **Spaces:** Collaborative AI applications and public demos powered by the community.
- **Enterprise Solutions:** Tailored team plans, enterprise support, inference endpoints, and storage buckets to enable scalable AI operations for businesses.
- **HuggingChat:** An open, community-driven chat interface built on cutting-edge language models.

---

## Company Culture

- **Community-Driven:** Hugging Face thrives on open collaboration and openly accessible AI, encouraging transparency and ethical AI development.
- **Innovative:** With a talented science and engineering team exploring the edge of technology, Hugging Face continually pushes forward in research, tools, and AI applications.
- **Inclusive:** The platform supports a diverse global community from hobbyists to industry experts, emphasizing continuous learning and sharing.
- **Open Source Leadership:** Hugging Face is a strong proponent of open-source software, hosting some of the most utilized ML libraries globally.

---

## Our Customers

Hugging Face serves a wide spectrum of users including:

- **Machine Learning Engineers & Researchers**  
- **AI Startups & Enterprises** leveraging ML models and pipelines in production.
- **Academia & Students** seeking to learn and apply AI through accessible datasets and models.
- **Developers and AI Enthusiasts** building AI applications and contributing to community projects.

Enterprise clients benefit from premium support, dedicated inference infrastructure, and secure storage solutions designed to seamlessly integrate ML models at scale.

---

## Careers at Hugging Face

Join an exciting AI frontier company that is:

- Driving open and ethical AI solutions worldwide.
- Committed to diversity and an inclusive work culture.
- Offering opportunities to work with a multidisciplinary team of researchers, engineers, and product experts.
- Enabled with flexible roles in engineering, research science, product management, and community engagement.

Interested candidates can explore and apply for open positions via the Careers page on the Hugging Face website.

---

## Connect and Learn

- **Community Forums & Discord:** Engage with peers and experts.
- **Blog & Daily Papers:** Stay updated on AI research and company news.
- **GitHub:** Access and contribute to open-source repositories.
- **Documentation & Learning Resources:** Tutorials and guides to help accelerate your AI journey.

---

## Brand Identity

Hugging Face’s friendly and approachable visual identity reflects the company’s mission to make AI accessible and vibrant. The brand colors (#FFD21E, #FF9D00, #6B7280) and logos are available for partners and collaborators to use.

---

## Discover Hugging Face Today

Explore the future of AI through collaboration, innovation, and open-source excellence at:

**https://huggingface.co**

Become part of a fast-growing community that is shaping the future of machine learning.

---

*Hugging Face — The Home of Machine Learning.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [18]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response),
        display_id=display_handle.display_id)

In [19]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for URL: https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a leading AI community dedicated to building the future of machine learning. It serves as a collaborative platform where millions of machine learning practitioners, researchers, and developers come together to create, discover, and share models, datasets, and applications.

The company offers access to over **2 million models**, **500,000+ datasets**, and a rich ecosystem of AI apps and services, fostering an open environment where AI innovation thrives.

---

## What Hugging Face Offers

- **Models**: A vast repository with cutting-edge models updated regularly, covering numerous AI tasks including natural language processing, computer vision, video generation, and more.
- **Datasets**: Extensive collections of datasets across various domains such as cybersecurity, finance, video, and language resources.
- **Spaces**: Hosting and deploying AI applications that allow the community and enterprises to run and showcase models interactively.
- **Enterprise Solutions**: Tailored plans including Hugging Face PRO, enterprise support, inference providers, inference endpoints, and secure storage buckets to meet business needs.
- **Community Resources**: Active community interaction through forums, Discord, GitHub, daily papers, blogs, and learning platforms to keep users at the forefront of AI advancements.

---

## Company Culture

Hugging Face promotes a vibrant, open, and collaborative culture centered around:

- **Community-first mindset**: Building an inclusive environment where ML experts and enthusiasts contribute and share knowledge freely.
- **Innovation-driven**: Empowering users by continually expanding the ecosystem with new models, datasets, and technological solutions.
- **Transparency and Accessibility**: Making AI resources publicly accessible while offering enterprise-grade solutions to businesses.
- **Learning and Growth**: Supporting continuous education through extensive documentation, forums, blogs, and active discussions.

---

## Customers

Hugging Face serves a diverse range of users including:

- Individual machine learning developers and researchers.
- AI startups and independent teams leveraging state-of-the-art models.
- Enterprises seeking scalable AI solutions with professional support and infrastructure.
- Academic and research institutions utilizing vast datasets and collaborative tools.

---

## Careers and Opportunities

Hugging Face is continually growing and offers exciting career opportunities for passionate individuals who want to be part of the AI revolution. The company values talent who are:

- Enthusiastic about open-source AI and its impact.
- Skilled in machine learning, NLP, software engineering, research, and data science.
- Collaborative and eager to contribute to a thriving AI community.
- Motivated by challenges in building cutting-edge AI tools, products, and infrastructure.

Prospective applicants can explore roles in engineering, research, community management, and enterprise customer support.

---

## Why Choose Hugging Face?

- Access to one of the largest and most active machine learning communities globally.
- Robust, cutting-edge tools and infrastructure to accelerate AI development.
- Flexible solutions ranging from open platform access to enterprise-grade products.
- Commitment to open science and collaborative innovation moving AI forward.

---

## Connect and Explore

- Visit: https://huggingface.co
- Join the community on Discord, GitHub, and forums.
- Explore models, datasets, and AI apps.
- Learn and stay updated with daily papers and blog posts.

---

**Hugging Face**: The AI community where tomorrow’s machine learning is created today.